In [29]:
import torch
from torch import nn
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [30]:
num_classes=10
device='cuda'

In [31]:
def get_model(num_classes):
    """Create Faster R-CNN model"""
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features,10)
    return model

In [ ]:
model = get_model(10).to('cpu')

c:\Users\main\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\main\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import cv2
import torch
import numpy as np
import time
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Load model weights using the correct map_location
state_dict = torch.load(r"D:\Nus internship\Intermediate\project 2\final_model2.pth", map_location='cuda')
model.load_state_dict(state_dict)
model.eval()
model.to('cuda')

# Class dictionary
class_dict = {
    0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask',
    4: 'NO-Safety Vest', 5: 'Person', 6: 'Safety Cone',
    7: 'Safety Vest', 8: 'machinery', 9: 'vehicle'
}

# Preprocessing
def preprocess_frame(frame):
    frame_resized = cv2.resize(frame, (512,512))
    img = frame_resized.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # HWC → CHW
    img_tensor = torch.tensor(img).unsqueeze(0).to('cuda')
    return img_tensor, frame_resized

# Decode output
def decode_output(output, thresh=0.5):
    boxes = output['boxes'].detach().cpu().numpy()
    scores = output['scores'].detach().cpu().numpy()
    labels = output['labels'].detach().cpu().numpy()
    keep = scores >= thresh
    return boxes[keep], scores[keep], labels[keep]

# Draw boxes
# Draw boxes
def draw_boxes(frame, boxes, labels, scores):
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = map(int, box)
        label_text = class_dict.get(label, f"Class {label}")
        text = f"{label_text} @{score:.2f}"

        # Set color based on whether it's unsafe
        color = (0, 255, 0)  # Green (safe)
        if label_text in unsafe_classes:
            color = (0, 0, 255)  # Red (unsafe)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return frame

# Unsafe class labels to trigger screenshot
unsafe_classes = {'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest'}

# Webcam stream
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Could not open webcam.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img_tensor, resized_frame = preprocess_frame(frame)

    with torch.no_grad():
        output = model(img_tensor)[0]
        boxes, scores, labels = decode_output(output, thresh=0.5)

    # Check for unsafe classes
    detected_unsafe = False
    for label in labels:
        class_name = class_dict.get(label, "")
        if class_name in unsafe_classes:
            detected_unsafe = True
            break

    # Save screenshot if any unsafe class is detected
    if detected_unsafe:
        timestamp = time.strftime("%Y-%m%d-%H-%M-%S")
        filename = f"/Users/chowdaryadithyasaividivada/Documents/unsafe/unsafe_detected_{timestamp}.png"
        cv2.imwrite(filename, resized_frame)
        print(f"📸 Screenshot saved: {filename}")

    frame_with_boxes = draw_boxes(resized_frame, boxes, labels, scores)
    cv2.imshow("🔍 Live Object Detection", frame_with_boxes)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.